[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_01_Introduction/M1_04_sample_data_exploration.ipynb)

# 📊 Module 01: Sample Data Exploration

**Purpose**: Get hands-on with real data! Load, explore, visualize, and define the prediction problem.

**Module**: Module 01 - Introduction  
**Author**: Ruby van Rooyen  
**Date**: 2025-12-30

---

## 📋 Overview

In this notebook, you will:
- [ ] Load a sample dataset (bikes + weather)
- [ ] Inspect data structure and quality
- [ ] Perform basic exploratory data analysis (EDA)
- [ ] Create simple visualizations
- [ ] Identify patterns and insights
- [ ] Define the prediction problem clearly
- [ ] Set project goals and success metrics

**Goal**: Build confidence with real data and understand what we're trying to predict!

---

## 🔧 Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import sys
import os
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
    # Clone the repository if needed
    import os
    if not os.path.exists('bike-availability-data-science-full'):
        !git clone https://github.com/vinculum3141-ship-it/bike-availability-data-science-full.git
    # Set project root
    project_root = '/content/bike-availability-data-science-full'
    os.chdir(project_root)
else:
    print("📍 Running locally")
    # Get notebook directory (where this .ipynb file is located)
    notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    # Get project root (two levels up from notebooks/Module_01_Introduction)
    project_root = os.path.abspath(os.path.join(notebook_dir, '../..'))
    # Add to path for imports
    if project_root not in sys.path:
        sys.path.append(project_root)
    print(f"📂 Project root: {project_root}")

print(f"📁 Working directory: {os.getcwd()}")
print(f"📍 Data will be loaded from: {project_root}")

print("✅ Setup complete!")
print(f"🐍 Python version: {sys.version.split()[0]}")

### 📚 Key Libraries in This Notebook

**Why These Libraries?**

**🐼 Pandas** - Data manipulation and analysis
- Provides DataFrame structures for handling tabular data efficiently
- Offers powerful data cleaning, transformation, and aggregation functions
- Handles missing data, merging datasets, and time series operations
- Simplifies data loading from various formats (CSV, Excel, SQL, etc.)
- Makes exploratory data analysis faster with built-in statistical methods

**📊 Seaborn** - Statistical visualization
- Built on matplotlib for creating publication-quality plots with minimal code
- Integrates seamlessly with pandas DataFrames
- Provides high-level functions for visualizing distributions, relationships, and patterns
- Includes built-in themes and color palettes for better aesthetics
- Automatically handles common visualization tasks (correlations, distributions, categorical comparisons)

**🔢 NumPy** - Numerical computing
- Foundation for scientific computing in Python
- Fast array operations and mathematical functions
- Used internally by pandas and many ML libraries

**📈 Matplotlib** - Base plotting library
- Low-level control for customized visualizations
- Seaborn builds on top of matplotlib
- Essential for fine-tuning plots

**Workflow**: **Pandas** for data wrangling → **Seaborn/Matplotlib** for visualization → Insights for feature engineering and modeling decisions! 🚀

---

## 📥 Part 1: Load Sample Data

Let's load a sample dataset that combines bike availability and weather data.

This dataset includes:
- **Timestamps** - Date and time of observations
- **Station information** - Station ID, name, location
- **Bike availability** - Number of bikes available (our target!)
- **Weather conditions** - Temperature, precipitation, wind
- **Temporal features** - Hour, day of week, weekend flag

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. LOAD DATA
# ═══════════════════════════════════════════════════════════

# Build data path using project_root for cross-platform consistency
data_path = os.path.join(project_root, 'data', 'raw', 'sample_bike_weather.csv')

print("📥 Loading sample data...\n")
print(f"📂 Data path: {data_path}")

# Verify file exists before attempting to load
if not os.path.exists(data_path):
    print(f"\n⚠️  File not found at: {data_path}")
    print(f"\n🔍 Checking alternate locations...")
    # Try relative path from current directory
    alt_path = os.path.join(os.getcwd(), 'data', 'raw', 'sample_bike_weather.csv')
    if os.path.exists(alt_path):
        data_path = alt_path
        print(f"✅ Found at: {data_path}")
    else:
        print(f"❌ File not found. Please check:")
        print(f"   1. Current directory: {os.getcwd()}")
        print(f"   2. Project root: {project_root}")
        print(f"   3. File exists at: {data_path}")
        raise FileNotFoundError(f"Cannot locate sample_bike_weather.csv")

try:
    df = pd.read_csv(data_path, parse_dates=['timestamp'])
    print(f"\n✅ Data loaded successfully!")
    print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"📅 Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    
except Exception as e:
    print(f"\n❌ Error loading data: {str(e)}")
    print("\n💡 Troubleshooting:")
    print("   - If in Colab: Make sure repository was cloned successfully")
    print("   - If local: Verify you're running from the project directory")
    print(f"   - Expected file location: {data_path}")
    raise

---

## 🔍 Part 2: Initial Data Inspection

Let's take a first look at the data structure.

### First Look: Display Sample Rows

### 🕵️ Why Data Inspection Matters

**This isn't busywork - it's essential detective work!**

Data inspection is the **first critical step** in any data science workflow. Here's why we always start here:

**1. ✅ Verify Successful Loading**
   - Confirm the file loaded correctly and contains expected data
   - Catch loading errors early before they cause downstream problems

**2. 🏗️ Understand Structure**
   - See what columns actually exist (not just what we expect)
   - Understand row counts and general shape
   - Get a "feel" for the data before diving deeper

**3. 🔤 Identify Data Types**
   - **Critical** because wrong types cause errors later
   - Dates stored as strings won't allow time-based operations
   - Numbers as strings won't allow mathematical calculations
   - Affects memory usage and performance

**4. 🚨 Spot Problems Early**
   - Missing values (nulls, NaN)
   - Unexpected data formats
   - Encoding issues
   - Duplicate columns
   - **Much cheaper to fix now than after hours of analysis!**

**5. 🧠 Build Mental Model**
   - Understand what you're working with
   - Plan data cleaning and transformations
   - Identify which features might be useful

**6. 🎯 Plan Next Steps**
   - What cleaning is needed?
   - What transformations make sense?
   - What visualizations would be informative?

**The Data Science Workflow:**
```
Load → Inspect → Understand → Clean → Analyze → Model
         ↑
    YOU ARE HERE!
```

**Pro Tip**: Skipping inspection is like starting to build without reading the blueprints - you'll likely go in the wrong direction and waste time! Always inspect first. 🔍

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. DISPLAY FIRST ROWS
# ═══════════════════════════════════════════════════════════

print("📋 First 10 rows of the dataset:\n")
df.head(10)

### Data Structure: Column Types and Info

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. DATA INFO
# ═══════════════════════════════════════════════════════════

print("📊 Dataset Information:\n")
print(df.info())

### Data Types: What Do We Have?

Let's understand each column:

| Column | Type | Description |
|--------|------|-------------|
| `timestamp` | datetime | Date and time of observation |
| `station_id` | string | Unique station identifier |
| `station_name` | string | Human-readable station name |
| `latitude` | float | Station latitude coordinate |
| `longitude` | float | Station longitude coordinate |
| `bikes_available` | int | **TARGET** - Number of bikes available |
| `docks_available` | int | Number of empty docking spaces |
| `temperature` | float | Temperature in °C |
| `precipitation` | float | Precipitation in mm |
| `windspeed` | float | Wind speed in km/h |
| `hour` | int | Hour of day (0-23) |
| `day_of_week` | int | Day of week (0=Monday, 6=Sunday) |
| `is_weekend` | int | Weekend indicator (0=weekday, 1=weekend) |

### Data Quality: Explicit Checks

Let's explicitly check for common data quality issues that are easy to miss!

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4A. DATA QUALITY CHECKS
# ═══════════════════════════════════════════════════════════

print("🔍 DATA QUALITY ASSESSMENT\n")
print("=" * 60)

# 1. Missing Values
print("\n1️⃣ MISSING VALUES:")
print("-" * 60)
null_counts_per_column = df.isnull().sum()
null_percentage_per_column = (df.isnull().sum() / len(df) * 100).round(2)
missing_data_summary = pd.DataFrame({
    'Missing Count': null_counts_per_column,
    'Missing %': null_percentage_per_column
})
print(missing_data_summary[missing_data_summary['Missing Count'] > 0])

if null_counts_per_column.sum() == 0:
    print("✅ No missing values detected!")
else:
    print(f"⚠️  Total missing values: {null_counts_per_column.sum()}")

# 2. Duplicate Rows
print("\n2️⃣ DUPLICATE ROWS:")
print("-" * 60)
total_duplicate_rows = df.duplicated().sum()
if total_duplicate_rows == 0:
    print("✅ No duplicate rows detected!")
else:
    print(f"⚠️  {total_duplicate_rows} duplicate rows found ({total_duplicate_rows/len(df)*100:.2f}%)")

# 3. Data Type Verification
print("\n3️⃣ DATA TYPE CHECK:")
print("-" * 60)
print(df.dtypes)
print("\n✅ Verify that:")
print("   - timestamp is datetime64")
print("   - Numeric columns are int64/float64")
print("   - Categorical columns are object/string")

# 4. Value Ranges (for key columns)
print("\n4️⃣ VALUE RANGE CHECKS:")
print("-" * 60)
print(f"bikes_available: min={df['bikes_available'].min()}, max={df['bikes_available'].max()}")
print(f"temperature:     min={df['temperature'].min():.1f}°C, max={df['temperature'].max():.1f}°C")
print(f"precipitation:   min={df['precipitation'].min():.1f}mm, max={df['precipitation'].max():.1f}mm")

# Check for negative values where they shouldn't exist
if (df['bikes_available'] < 0).any():
    print("⚠️  WARNING: Negative bike counts detected!")
if (df['temperature'] < -50).any() or (df['temperature'] > 50).any():
    print("⚠️  WARNING: Suspicious temperature values!")
if (df['precipitation'] < 0).any():
    print("⚠️  WARNING: Negative precipitation detected!")
    
print("\n✅ Data quality assessment complete!")
print("=" * 60)

---

## 🧠 Strategic Thinking: From Inspection to Action

**Now that we've inspected our data, what's next?**

You need a **thinking framework** to move from "I see the data" to "I know what to do with it."

### 🎯 The Core Questions

When planning your analysis, always ask:

1. **Cleaning:** What problems need fixing? Why are they problems?
2. **Transformations:** What does my model need? (Scaling? Encoding?)
3. **Features:** What information predicts my target?
4. **Visualizations:** What patterns am I looking for?
5. **Iteration:** What did I learn? What should I try next?

---

### 📋 Quick Decision Guide

**Missing Values:**
- <5% missing → Drop rows
- 5-30% missing → Impute (mean/forward-fill for time series)
- >30% missing → Investigate why, consider dropping feature

**Outliers:**
- Domain valid? → Keep them!
- Data error? → Remove or cap
- Suspicious? → Investigate first

**Feature Selection:**
- ✅ Keep: Correlated with target, domain relevant, available at prediction time
- ❌ Drop: No variance, perfect correlation with others, data leakage

**Model Requirements:**
- Tree-based (Random Forest, XGBoost) → No scaling needed ✅
- Linear models (Ridge, Lasso) → Scale features! ⚠️
- Neural Networks, KNN → Scale CRITICAL! 🔴

---

### 💡 Pro Tips for Bike Availability

**For our specific problem:**

1. **Temporal features matter most**
   - Hour, day of week, weekend indicator
   - Lag features (bikes 1 hour ago)
   - Rolling averages

2. **Weather is secondary but helpful**
   - Temperature, precipitation affect usage
   - Consider interactions (weather × time)

3. **Watch for data leakage!**
   - ❌ Don't use `docks_available` (inverse of `bikes_available`)
   - ❌ Don't use future information to predict past

4. **Start simple, iterate**
   - Basic features first → See what works → Add complexity

---

### 📖 Want the Full Framework?

**For comprehensive guidance on:**
- Detailed cleaning decision matrices
- Transformation timing and strategies
- Feature engineering techniques
- Visualization selection guide
- The iterative data science process
- Common mistakes and how to avoid them

**See:** [DATA_SCIENCE_THINKING_FRAMEWORK.md](../../docs/guides/DATA_SCIENCE_THINKING_FRAMEWORK.md)

**🎯 As you work through this notebook, keep asking:**
- *Why am I doing this?*
- *What question does it answer?*
- *What pattern am I looking for?*
- *How does this inform my modeling decisions?*

**Think critically, not mechanically!** 🧠

---

## 🤖 Model Requirements Quick Reference

**Different models have different data needs.** Here's what you need to know:

### Scaling Requirements by Model Type

| Model Type | Scaling Needed? | Why? |
|------------|----------------|------|
| **Random Forest, XGBoost, Decision Trees** | ❌ **NO** | Trees split on values, not distances. Scale doesn't matter. |
| **Ridge, Lasso, Linear Regression** | ✅ **Helpful** | Coefficients interpret magnitude as importance. |
| **KNN, SVM, Neural Networks** | 🔴 **CRITICAL** | Distance-based. Large-scale features dominate! |

---

### 🎯 Recommended Starting Point for This Project

**1st Choice:** **Random Forest** or **XGBoost**
- ✅ No scaling needed
- ✅ Handles mixed data types
- ✅ Robust to outliers
- ✅ Feature importance scores
- ✅ Great performance out-of-the-box

**2nd Choice:** **Ridge/Lasso**
- ✅ Interpretable coefficients
- ✅ Automatic feature selection (Lasso)
- ⚠️ Requires scaling and encoding
- ⚠️ Assumes linearity

**Avoid Initially:** KNN, Neural Networks
- ❌ Need extensive preprocessing
- ❌ Require more data
- ❌ Harder to tune

---

### ⚠️ Data Preparation Priorities

**Must Do:**
- 🔴 Handle missing values (all models)
- 🔴 Engineer temporal features (critical for time series)
- 🔴 Avoid data leakage (don't use `docks_available`!)

**For Linear/Distance-Based Models Only:**
- 🟡 Scale numeric features
- 🟡 Encode categorical variables
- 🟢 Handle outliers

---

### 📖 Want Model Details?

**For comprehensive information on:**
- 7 regression model comparison (pros/cons/requirements)
- Classification vs regression decision guide
- Unsupervised learning options
- Time series specific models
- Real-world examples and use cases

**See:** [ML_MODEL_TYPES_REFERENCE.md](../../docs/guides/ML_MODEL_TYPES_REFERENCE.md)

**💡 As you explore data, ask:** *"How will this feature work with different models?"*

**Next**: Let's visualize our data with these requirements in mind!

---

## 📈 Part 3: Descriptive Statistics

Let's get statistical summaries of our numerical columns.

### 📊 Understanding `df.describe()`

**What does it do?**

`df.describe()` generates **8 key statistics** for each numeric column in one line:

| Statistic | What It Tells You | Why It Matters |
|-----------|-------------------|----------------|
| **count** | Number of non-null values | Detect missing data |
| **mean** | Average value | Central tendency |
| **std** | Standard deviation | How spread out the data is |
| **min** | Minimum value | Lower bound, spot outliers |
| **25%** | First quartile (Q1) | 25% of data is below this |
| **50%** | Median (Q2) | Middle value (robust to outliers) |
| **75%** | Third quartile (Q3) | 75% of data is below this |
| **max** | Maximum value | Upper bound, spot outliers |

**What you can spot instantly:**
- 🔍 **Outliers:** min/max far from mean
- 📐 **Skewness:** mean ≠ median (50%)
- 📏 **Scale differences:** hour (0-23) vs temperature (-10 to 30)
- 🚨 **Missing data:** count < total rows
- 📊 **Low variance:** std near 0 (feature won't help prediction)

**Will it fail on non-numeric columns?**

❌ **No!** `df.describe()` is smart:
- ✅ **Automatically includes only numeric columns** (int, float)
- ✅ **Silently skips** strings, dates, objects - no error!

**Pro Tips:**
```python
df.describe()              # Only numeric columns (default)
df.describe(include='all') # Include ALL columns (numeric + non-numeric)
df.describe(include='object') # Only text/categorical columns
```

For non-numeric columns, you get different stats: `count`, `unique`, `top` (most frequent), `freq`.

**🎯 Look for:** Anything unexpected! Outliers, weird ranges, or suspicious values that need investigation.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. DESCRIPTIVE STATISTICS
# ═══════════════════════════════════════════════════════════

print("📊 Descriptive Statistics:\n")
df.describe()

### 🔍 What These Statistics Tell Us

**Bike Availability:**
- Mean: ~11 bikes available on average
- Range: 0-20 bikes (some stations run completely empty!)
- This is what we want to predict!

**Weather Conditions:**
- Temperature: ~8°C average (winter data)
- Precipitation: ~0.4mm average (mostly dry)
- Wind speed: ~9.5 km/h average

**Temporal Patterns:**
- Data covers various hours (6-20)
- Both weekdays and weekends included

---

## 🏷️ Part 4: Categorical Analysis

Let's look at the categorical/discrete variables.

### 🔤 Why Different Methods for Categoricals?

**Categorical vs Numeric - A Critical Distinction:**

| Numeric Data (Part 3) | Categorical Data (Part 4) |
|-----------------------|---------------------------|
| Continuous values | Discrete categories/labels |
| **Math operations make sense** | **Counting makes sense** |
| Mean, median, std are meaningful | Frequency, unique counts are meaningful |
| Example: Temperature = 15°C (can average) | Example: Station = "Central" (can't average!) |

---

### 🛠️ Categorical Analysis Methods

**We use these three key methods:**

| Method | What It Returns | Why It Matters | Example |
|--------|-----------------|----------------|---------|
| **`.nunique()`** | Count of unique values | **Cardinality check**<br>High cardinality (>100) may need grouping | `df['station_id'].nunique()` → 3 or 1000? |
| **`.unique()`** | Array of actual unique values | **See the categories**<br>Understand what values exist | `df['station_name'].unique()` → ["Central", "Mall", "Park"] |
| **`.value_counts()`** | Frequency of each value | **Find imbalances**<br>Rare categories? Dominant ones? | Weekday: 500, Weekend: 50 ← Imbalanced! |

---

### ⚠️ Why This Matters for ML

**1. High Cardinality Problem:**
- 1000 unique station IDs? → Model complexity explodes, overfitting risk
- **Solution:** Group rare categories, use embeddings, or aggregate

**2. Imbalanced Categories:**
- One category dominates (90%), others rare (1%)? → Model won't learn rare ones
- **Solution:** Collect more data, resample, or use stratified sampling

**3. Data Quality Issues:**
- Unexpected values? `[0, 1, 2, 'yes']` in binary field? → Inconsistent encoding
- **Solution:** Clean and standardize before modeling

---

### ❌ What You CAN'T Do

**Don't calculate statistics that don't make sense:**

```python
# ❌ WRONG - Can't average text!
df['station_name'].mean()  # Error!

# ❌ MEANINGLESS - Even if numeric, day codes aren't quantities
df['day_of_week'].std()    # What does "standard deviation of Monday/Tuesday" mean?
```

**✅ DO count and examine:**
```python
# ✅ RIGHT - How many unique stations?
df['station_name'].nunique()

# ✅ RIGHT - What are the station names?
df['station_name'].unique()

# ✅ RIGHT - How often does each appear?
df['station_name'].value_counts()
```

---

### 🎯 Quick Decision Rule

**Ask:** *"Can I meaningfully average this?"*

- **YES** → It's numeric → Use `df.describe()` (Part 3)
- **NO** → It's categorical → Use `.nunique()`, `.unique()`, `.value_counts()` (Part 4)

**🔍 What to look for:** High cardinality, imbalanced categories, or unexpected values that need attention!

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. CATEGORICAL ANALYSIS
# ═══════════════════════════════════════════════════════════

print("🏷️ Unique Values:\n")
print(f"Number of unique stations: {df['station_id'].nunique()}")
print(f"Station names: {df['station_name'].unique()}")
print(f"\nNumber of unique timestamps: {df['timestamp'].nunique()}")
print(f"Hours covered: {sorted(df['hour'].unique())}")
print(f"Days of week: {sorted(df['day_of_week'].unique())}")
print(f"\nWeekday vs Weekend distribution:")
print(df['is_weekend'].value_counts().to_dict())

---

## 🎨 Part 5: Data Visualization

Now let's visualize the data to identify patterns!

### Visualization 1: Bike Availability Over Time

**🎯 What This Visualization Shows:**

A **time series line plot** that tracks how bike availability changes over time at each station. Each colored line represents a different station, allowing us to see both individual station patterns and compare across locations.

---

### 🔍 Pattern Identification Goals

**What we're looking for:**

| Pattern Type | What to Observe | Why It Matters for ML |
|-------------|-----------------|----------------------|
| **📈 Trends** | Is availability increasing/decreasing over time? | Indicates temporal dependencies → need time features |
| **🔄 Cycles** | Do patterns repeat regularly (daily, hourly)? | Regular cycles → temporal features will be strong predictors |
| **🎪 Station Differences** | Do different stations behave differently? | Different patterns → `station_id` is important feature |
| **🚨 Zero Events** | When do lines hit y=0 (empty stations)? | **This is the business problem!** Must predict these |
| **〰️ Volatility** | Smooth vs jagged lines? | High variability → harder to predict, need more features |

---

### 💡 How This Informs Modeling Decisions

**Based on what you see, you'll make key decisions:**

**If you observe strong hourly patterns:**
- → `hour` will be a top feature
- → Consider cyclical encoding (sin/cos) for hour of day
- → May need hour × station interactions

**If stations behave very differently:**
- → `station_id` is critical
- → May need station-specific models
- → Consider station characteristics (location, capacity)

**If availability regularly drops to zero:**
- → Model must handle edge cases well
- → Consider classification first (empty vs not empty)
- → Prioritize preventing false negatives (predicting bikes when there are none)

**If patterns look random/chaotic:**
- → Need more features (weather, events, holidays)
- → May need complex models (ensemble methods)
- → Consider external data sources

---

### 🎯 What to Look For

**As you view the plot, ask yourself:**

1. **Predictability:** Do patterns look regular or random?
   - Regular → Easier problem, temporal features sufficient
   - Random → Need more feature engineering

2. **Consistency:** Are patterns similar across days?
   - Yes → Model will generalize well
   - No → May need more data or context features

3. **Extreme values:** Do stations hit capacity limits (max/min)?
   - Yes → Consider bounded regression or classification

4. **Problem severity:** How often do stations empty?
   - Frequent → Urgent business problem
   - Rare → May be acceptable with simple rebalancing

**Remember:** This isn't just a visualization—it's your **first evidence of what drives bike availability**, which directly shapes what features you'll engineer and what models you'll choose! 📊

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. TIME SERIES PLOT
# ═══════════════════════════════════════════════════════════

# Create figure
plt.figure(figsize=(14, 6))

# Plot each station separately
for station in df['station_name'].unique():
    station_data = df[df['station_name'] == station]
    plt.plot(station_data['timestamp'], 
             station_data['bikes_available'], 
             marker='o', 
             label=station,
             linewidth=2,
             markersize=6,
             alpha=0.8)

plt.title('Bike Availability Over Time by Station', fontsize=16, fontweight='bold')
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('Bikes Available', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 What do you notice?")
print("   - Different patterns between stations")
print("   - Temporal trends (time of day matters!)")
print("   - Some stations reach zero bikes (problem!)")

### 🤔 Reflection: Time Series Visualization

**Stop and think! Answer these questions before moving on:**

**1. Why did I create this visualization?**
<details>
<summary>Click to see example answer</summary>

To understand how bike availability changes over time and identify temporal patterns that might be predictive. Time series plots reveal trends, seasonality, and station-specific behaviors.
</details>

**2. What question does it answer?**
<details>
<summary>Click to see example answer</summary>

- Do different stations have different availability patterns?
- Are there clear time-of-day trends?
- Do stations ever run out of bikes (reach zero)?
- Is availability stable or highly variable?
</details>

**3. What patterns am I looking for?**
<details>
<summary>Click to see example answer</summary>

- **Trends**: Increasing/decreasing availability over time
- **Cycles**: Regular patterns (hourly, daily)
- **Volatility**: How much availability fluctuates
- **Zero events**: When stations run empty (critical for business!)
- **Station differences**: Some stations busier than others
</details>

**4. How does this inform my modeling decisions?**
<details>
<summary>Click to see example answer</summary>

- **Feature engineering**: Need temporal features (hour, day, lags)
- **Model choice**: Time patterns suggest temporal features will be important
- **Problem scope**: Zero bikes are common → model must predict these well
- **Station-specific models?**: Different patterns suggest station ID is important
- **Data requirements**: Need sufficient history to capture patterns
</details>

---

**✍️ Your observations** (write your insights here):
```
[What patterns did YOU notice? What surprised you?]






```

---

### Visualization 2: Bike Availability by Hour of Day

**🎯 What This Visualization Shows:**

A **bar chart** displaying the average bike availability for each hour of the day. Each bar represents the mean number of bikes available across all stations and days for that specific hour, revealing time-of-day patterns.

---

### 🔍 Pattern Identification Goals

**What we're looking for:**

| Pattern Type | What to Observe | Why It Matters for ML |
|-------------|-----------------|----------------------|
| **⏰ Rush Hour Effects** | Lower bars during commute times (7-9am, 5-7pm) | People take bikes to work/home → `hour` will be a **top predictor** |
| **📊 Peak vs Off-Peak** | Highest bars = most bikes available | Identifies best times for rebalancing operations |
| **📏 Magnitude of Variation** | How much do bars vary in height? | Large variation → hour is very important feature |
| **〰️ Non-Linear Patterns** | Bars jump around vs smooth progression | Non-linear → tree-based models better than linear |
| **✅ Predictability Signal** | Regular pattern vs irregular | Regular → easy to predict; irregular → need more features |

---

### 💡 How This Informs Modeling Decisions

**If you see a strong pattern (big differences between hours):**
- → **`hour` is a critical feature** - must include it!
- → Consider creating **derived features:**
  - `is_rush_hour` (7-9am or 5-7pm)
  - `is_business_hours` (9am-5pm)
  - `time_of_day_category` (morning/afternoon/evening/night)

**If pattern is non-linear (not a straight slope):**
- → **Tree-based models** (Random Forest, XGBoost) handle this naturally
- → For **linear models**, need:
  - Polynomial features (`hour²`, `hour³`)
  - One-hot encoding (treat each hour separately)
  - Cyclical encoding (sin/cos transformations)

**If some hours are critical (very low availability):**
- → **Prioritize those hours** in evaluation
- → Consider separate models for rush vs non-rush
- → Use weighted loss functions to penalize rush hour errors more

**If weekday vs weekend patterns differ:**
- → Need **interaction features:** `hour × is_weekend`
- → 8am Monday (rush) ≠ 8am Saturday (leisure)!

---

### 🎯 What to Look For

**As you view the chart, ask yourself:**

1. **Is there a clear pattern?**
   - Yes → Hour is predictive, model will learn it
   - No → Time might not matter, or need more granular data

2. **Which hours are problematic?**
   - Lowest bars = potential empty station times
   - **These are what we're trying to predict and prevent!**

3. **Is the pattern what you'd expect?**
   - Expected (low at 8am rush) → Good! Domain logic validates
   - Unexpected (high at 3am?) → Data quality issue or interesting insight?

4. **How much does hour matter?**
   - Range 15 → 5 bikes → **Huge effect!** (hour is critical)
   - Range 11 → 9 bikes → Small effect (hour less important)

**Remember:** Time-of-day patterns are typically the **#1 or #2 most important feature** in bike-sharing prediction! This chart shows **when** bikes are available or scarce—a critical signal for predicting future availability. ⏰

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. HOURLY PATTERNS
# ═══════════════════════════════════════════════════════════

plt.figure(figsize=(12, 5))

# Average bikes by hour
hourly_avg = df.groupby('hour')['bikes_available'].mean()

plt.bar(hourly_avg.index, hourly_avg.values, color='steelblue', alpha=0.8, edgecolor='black')
plt.title('Average Bike Availability by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Average Bikes Available', fontsize=12)
plt.xticks(range(6, 21))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Pattern observed:")
print(f"   - Lowest availability: {hourly_avg.idxmin()}:00 ({hourly_avg.min():.1f} bikes)")
print(f"   - Highest availability: {hourly_avg.idxmax()}:00 ({hourly_avg.max():.1f} bikes)")
print("   - Morning rush vs evening patterns")

### 🤔 Reflection: Hourly Patterns

**Stop and think! Answer these questions before moving on:**

**1. Why did I create this visualization?**
<details>
<summary>Click to see example answer</summary>

To identify hour-of-day patterns in bike availability. Hour is likely a strong predictor since people have regular commute and activity patterns.
</details>

**2. What question does it answer?**
<details>
<summary>Click to see example answer</summary>

- Which hours have highest/lowest bike availability?
- Are there morning vs evening differences?
- Does this show commute patterns (rush hours)?
- Is the pattern consistent or highly variable by hour?
</details>

**3. What patterns am I looking for?**
<details>
<summary>Click to see example answer</summary>

- **Rush hour effects**: Lower availability during commute times (8-9am, 5-6pm)
- **Mid-day patterns**: Higher availability during work hours?
- **Evening trends**: How does availability change in evening?
- **Magnitude of change**: How much does hour matter?
</details>

**4. How does this inform my modeling decisions?**
<details>
<summary>Click to see example answer</summary>

- **Feature importance**: Hour will likely be a top predictor
- **Feature engineering**: Consider binning into categories (rush hour, mid-day, evening, night)
- **Non-linearity**: Pattern isn't linear → tree-based models might be better than linear
- **Interactions**: Hour × day_of_week might matter (different weekday/weekend patterns)
- **Model validation**: Need to preserve temporal structure in train/test split
</details>

---

**✍️ Your observations** (write your insights here):
```
[What hourly patterns did YOU see? Do they match your expectations?]






```

---

### Visualization 3: Temperature vs Bike Availability

**🎯 What This Visualization Shows:**

A **scatter plot** exploring the relationship between temperature and bike availability. Each point represents one observation, with different colors for different stations. This reveals whether weather (temperature) influences how many bikes are available.

---

### 🔍 Pattern Identification Goals

**What we're looking for:**

| Pattern Type | What to Observe | Why It Matters for ML |
|-------------|-----------------|----------------------|
| **📈 Linear Relationship** | Do points form a line (upward/downward)? | Positive/negative correlation → temperature is predictive |
| **💪 Correlation Strength** | How tight is the clustering? | Strong (>0.7) → major predictor; Weak (<0.3) → secondary |
| **〰️ Non-Linear Patterns** | Points form a curve vs straight line? | Curved → need polynomial features or tree-based models |
| **🎪 Station Differences** | Do colored clusters have different patterns? | Different slopes → need `temperature × station` interactions |
| **⚠️ Outliers** | Points far from trend? | Data quality issues or missing context features |

---

### 💡 How This Informs Modeling Decisions

**If correlation is weak (<0.3):**
- → Temperature alone isn't very predictive
- → **BUT** don't discard! May be important in **interactions**:
  - `temperature × hour` (cold mornings vs warm evenings)
  - `temperature × is_weekend` (weather matters more for leisure)
  - `temperature × precipitation` (combined weather effect)

**If correlation is moderate-to-strong (>0.3):**
- → Include temperature as a feature
- → Tree-based models: Use raw values
- → Linear models: May need scaling

**If pattern is non-linear (curved):**
- → **Tree-based models** capture automatically
- → **Linear models** need transformations:
  - Polynomial: `temperature²`, `temperature³`
  - Binning: `cold` (<10°C), `mild` (10-20°C), `warm` (>20°C)
  - Thresholds: `is_freezing`, `is_hot`

**If stations respond differently:**
- → Create interaction: `temperature × station_id`
- → Or build station-specific models

**If many outliers exist:**
- → Investigate: Special events? Holidays? Data errors?
- → May need additional context features

---

### 🎯 What to Look For

**As you view the plot, ask yourself:**

1. **Is there any relationship?**
   - Clear trend → Temperature is useful
   - Random cloud → Temperature alone won't help

2. **What's the direction?**
   - Upward slope → Warmer = more bikes available (less usage)
   - Downward slope → Warmer = fewer bikes (more usage)
   - **Typical:** Nice weather → more riding → fewer bikes left

3. **How strong is it?**
   - Check the correlation coefficient printed below
   - Weak (<0.3) → Secondary feature
   - Strong (>0.7) → Major predictor

4. **Is it linear or curved?**
   - Straight → Simple relationship
   - Curved → Complex, needs transformations

5. **Any surprising patterns?**
   - All one temperature? → Limited weather variety in data
   - Clear outliers? → Investigate those cases

**Remember:** For bike-sharing, temperature often has a **moderate effect**—it matters, but **time patterns are usually stronger**. The real value might come from temperature **interactions** with time and location features! 🌡️

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. WEATHER RELATIONSHIP
# ═══════════════════════════════════════════════════════════

plt.figure(figsize=(10, 6))

# Scatter plot with color by station
for station in df['station_name'].unique():
    station_data = df[df['station_name'] == station]
    plt.scatter(station_data['temperature'], 
                station_data['bikes_available'],
                label=station,
                s=100,
                alpha=0.6,
                edgecolors='black',
                linewidth=0.5)

plt.title('Bike Availability vs Temperature', fontsize=16, fontweight='bold')
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Bikes Available', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = df['temperature'].corr(df['bikes_available'])
print(f"📊 Correlation between temperature and bikes available: {correlation:.3f}")
if abs(correlation) < 0.3:
    print("   → Weak correlation")
elif abs(correlation) < 0.7:
    print("   → Moderate correlation")
else:
    print("   → Strong correlation")

### 🤔 Reflection: Weather Relationship

**Stop and think! Answer these questions before moving on:**

**1. Why did I create this visualization?**
<details>
<summary>Click to see example answer</summary>

To explore the relationship between weather (temperature) and bike availability. Weather likely influences cycling behavior, which affects availability patterns.
</details>

**2. What question does it answer?**
<details>
<summary>Click to see example answer</summary>

- Is there a relationship between temperature and bike availability?
- Is the relationship linear or non-linear?
- Does the relationship vary by station?
- How strong is the correlation?
</details>

**3. What patterns am I looking for?**
<details>
<summary>Click to see example answer</summary>

- **Linear trends**: Positive/negative correlation with temperature
- **Non-linear patterns**: Threshold effects (too cold → fewer rides)
- **Station differences**: Do all stations respond similarly to weather?
- **Outliers**: Unusual combinations of weather and availability
- **Strength**: Tight clustering (strong relationship) vs scattered (weak)
</details>

**4. How does this inform my modeling decisions?**
<details>
<summary>Click to see example answer</summary>

- **Feature relevance**: Correlation coefficient tells us if temperature matters
- **Weak correlation (<0.3)**: Temperature alone may not be predictive (but interactions might be!)
- **Non-linearity**: If pattern is curved → consider polynomial features or tree-based models
- **Interactions**: Temperature × hour or temperature × season might be important
- **Feature engineering**: Consider temperature bins (cold/mild/warm) or threshold indicators
- **Expectations**: Weather probably less important than temporal features for this problem
</details>

---

**✍️ Your observations** (write your insights here):
```
[What's the correlation? Strong or weak? What does this mean for your model?]






```

---

### Visualization 4: Weekday vs Weekend Comparison

**🎯 What This Visualization Shows:**

A **box plot** comparing bike availability between weekdays and weekends. Each box displays 5 key statistics (min, Q1, median, Q3, max) showing the distribution of bike availability for each category.

**📦 Box Plot Components:**
- **Box bottom:** 25th percentile (Q1) - 25% of data below this
- **Line in box:** Median (50th percentile) - middle value
- **Box top:** 75th percentile (Q3) - 75% of data below this
- **Whiskers:** Extend to min/max (or 1.5×IQR)
- **Dots:** Outliers beyond whiskers
- **Box height (IQR):** Spread of middle 50% of data

---

### 🔍 Pattern Identification Goals

**What we're looking for:**

| Pattern Type | What to Observe | Why It Matters for ML |
|-------------|-----------------|----------------------|
| **📊 Different Central Tendency** | Medians at different heights? | Different averages → `is_weekend` is useful feature |
| **📏 Different Variability** | One box taller than other? | Different predictability by category |
| **📐 Distribution Shape** | Where's median within box? | Skewed distributions suggest different processes |
| **⚠️ Outliers by Category** | More dots on weekday or weekend? | Different special event patterns |
| **🔄 Overlap Between Groups** | Do boxes overlap? | Degree of separation = feature importance |

---

### 💡 How This Informs Modeling Decisions

**If significant difference exists (medians far apart):**
- → **`is_weekend` is valuable** - include it!
- → Different usage patterns need different predictions
- → Business: Different rebalancing strategies for weekday/weekend

**If distributions have different spreads:**
- → **Consider interactions:** `hour × is_weekend`
- → Weekday 8am (rush) ≠ Weekend 8am (sleeping)
- → May need weighted predictions (higher uncertainty on variable days)

**If many outliers on one category:**
- → Investigate specific cases
- → May need additional features:
  - Holidays (look like weekends but labeled weekday)
  - Special events (festivals increase weekend variability)

**If minimal difference (boxes very similar):**
- → `is_weekend` alone may not be important
- → BUT interactions might still matter (`weekend × hour`)
- → Focus on other features

**If one category has wider box:**
- → Higher uncertainty for that category
- → Tree-based models handle automatically
- → Linear models may need variance weighting

---

### 🎯 What to Look For

**As you view the plot, ask yourself:**

1. **Is there a clear difference?**
   - Big separation → `is_weekend` is important
   - Heavy overlap → Less important feature

2. **Which has more bikes available?**
   - Weekday higher → Bikes accumulate (commuters travel TO stations)
   - Weekend higher → Less overall usage
   - Pattern reveals **usage behavior**

3. **How consistent are the patterns?**
   - Tall boxes → High variability, harder to predict
   - Short boxes → Consistent, easier to predict

4. **Are there outliers?**
   - Many → Investigate special cases
   - Few → Clean, consistent data

5. **Is the pattern expected?**
   - Expected: Weekdays have commute patterns
   - Expected: Weekends more leisure patterns
   - Unexpected? → Could be data quality or interesting insight

**Remember:** For bike-sharing, weekday vs weekend patterns almost always differ—commute patterns dominate weekdays, leisure patterns dominate weekends. The real modeling power comes from combining `is_weekend` with `hour` to capture these different temporal patterns! 📅

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. WEEKDAY VS WEEKEND
# ═══════════════════════════════════════════════════════════

plt.figure(figsize=(10, 6))

# Box plot
df['day_type'] = df['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})
sns.boxplot(data=df, x='day_type', y='bikes_available', hue='day_type', palette='Set2', legend=False)
plt.title('Bike Availability: Weekday vs Weekend', fontsize=16, fontweight='bold')
plt.xlabel('Day Type', fontsize=12)
plt.ylabel('Bikes Available', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Statistics
weekday_avg = df[df['is_weekend'] == 0]['bikes_available'].mean()
weekend_avg = df[df['is_weekend'] == 1]['bikes_available'].mean()

print(f"📊 Average bikes available:")
print(f"   Weekday: {weekday_avg:.1f} bikes")
print(f"   Weekend: {weekend_avg:.1f} bikes")
print(f"   Difference: {abs(weekday_avg - weekend_avg):.1f} bikes")

### 🤔 Reflection: Weekday vs Weekend

**Stop and think! Answer these questions before moving on:**

**1. Why did I create this visualization?**
<details>
<summary>Click to see example answer</summary>

To compare bike availability patterns between weekdays and weekends. Usage patterns differ (commute vs leisure), which should affect availability.
</details>

**2. What question does it answer?**
<details>
<summary>Click to see example answer</summary>

- Is there a significant difference between weekday and weekend availability?
- Which has higher/lower availability on average?
- Is the variability different (wider/narrower distributions)?
- Are there outliers in either group?
</details>

**3. What patterns am I looking for?**
<details>
<summary>Click to see example answer</summary>

- **Central tendency**: Different means/medians between groups
- **Spread**: Different variability (box widths, whisker lengths)
- **Outliers**: Unusual observations in either group
- **Distribution shape**: Skewed vs symmetric
- **Magnitude**: How big is the difference (practically significant)?
</details>

**4. How does this inform my modeling decisions?**
<details>
<summary>Click to see example answer</summary>

- **Feature importance**: If significant difference → `is_weekend` is a useful feature
- **Interactions**: Weekend × hour might capture different hourly patterns
- **Feature engineering**: Consider more granular time categories (weekday morning/evening, weekend)
- **Model expectations**: Tree-based models can split on this naturally; linear models need it encoded
- **Business insight**: Different rebalancing strategies for weekdays vs weekends
- **Small difference**: If minimal difference → maybe not as important as expected
</details>

---

**✍️ Your observations** (write your insights here):
```
[Is the difference large? Surprising? What does this tell you about bike usage patterns?]






```

---

### Visualization 5: Correlation Heatmap

**🎯 What This Visualization Shows:**

A **correlation heatmap** - a color-coded matrix displaying all pairwise correlations between numerical features. Each cell shows the Pearson correlation coefficient (-1 to +1) indicating how strongly two variables move together. This provides a comprehensive overview of relationships in your dataset at a glance.

**📊 Understanding Correlation Coefficients:**
- **+1.0:** Perfect positive correlation (both increase together)
- **0.0:** No linear relationship
- **-1.0:** Perfect negative correlation (one increases, other decreases)
- **Color coding:** Red = positive, Blue = negative, White = zero

---

### 🔍 Pattern Identification Goals

**What we're looking for:**

| Pattern Type | What to Observe | Why It Matters for ML |
|-------------|-----------------|----------------------|
| **🎯 Target Correlations** | Strong colors in `bikes_available` row/column | Identifies best single predictors (>0.5 = strong, <0.3 = weak) |
| **🔗 Multicollinearity** | High correlations between predictors (>0.7) | Problem for linear models (unstable coefficients, hard to interpret) |
| **✅ Feature Independence** | Near-white squares (≈0) between predictors | Features provide unique information (good!) |
| **❌ Redundant Features** | Very high correlations (>0.9) between predictors | One feature makes the other unnecessary → remove duplicate |
| **🔍 Unexpected Patterns** | Surprising correlations | Hidden relationships or data quality issues |

---

### 💡 How This Informs Modeling Decisions

**If strong target correlations exist (>0.5):**
- → Those features are **critical predictors** - must include!
- → Tree-based models will split on them early
- → Linear models will assign large coefficients
- → Focus feature engineering on these variables

**If all target correlations are weak (<0.3):**
- → No strong linear relationships exist
- → Need feature engineering: interactions, polynomials, binning
- → Consider non-linear models (trees outperform linear)
- → May need additional data sources

**If high multicollinearity between predictors (>0.7):**
- → **Linear models (Ridge, Lasso):**
  - Use Ridge regression (handles multicollinearity via regularization)
  - Or use Lasso (automatically selects one from correlated pair)
  - Or manually remove one from each correlated pair
- → **Tree-based models (Random Forest, XGBoost):**
  - No problem! Include all features
  - Trees naturally handle redundancy
- → **Feature engineering:**
  - Don't create interactions from highly correlated pairs (redundant)
  - Consider PCA if many intercorrelated features

**If features cluster into groups:**
- → Weather variables correlate with each other
- → Time variables correlate with each other
- → Each group provides distinct information type
- → Interactions between groups may be powerful (e.g., `temperature × hour`)

**If predictor correlation but no target correlation:**
- → Multiple features share information but none predict well
- → All are likely unhelpful
- → Consider removing entire group or using PCA

---

### 🎯 What to Look For

**As you view the heatmap, focus on these critical analyses:**

**1. The Most Important Row/Column: `bikes_available`**
   - Scan this row/column for darkest colors
   - Which features have strongest correlation (red or blue)?
   - These are your best single predictors
   - Prioritize these for feature engineering

**2. Multicollinearity Warnings:**
   - Dark squares between predictors (not involving target)
   - Example: `temperature` ↔ `hour` at 0.8 → Redundant for linear models
   - Decision needed: Keep both, remove one, or use regularization

**3. Feature Independence (Good Sign):**
   - Near-white squares = features provide unique information
   - Each adds distinct predictive power
   - Ideal for model performance

**4. Sign of Correlation:**
   - **Positive (red):** Both increase together
     - Example: `temperature` ↑ → `bikes_available` ↑ (less cycling when hot?)
   - **Negative (blue):** Inverse relationship
     - Example: `hour` ↑ (rush hour) → `bikes_available` ↓ (bikes in use)

**5. Magnitude vs Expectation:**
   - Does domain knowledge match the data?
   - Expected: Hour correlates strongly (time patterns matter)
   - Unexpected: `precipitation` shows no correlation? Check data quality!

**6. Symmetry Check:**
   - Heatmap should be symmetric (correlation is bidirectional)
   - Diagonal should be all 1.0 (each feature perfectly correlated with itself)
   - If not → Data or calculation error!

---

### ⚠️ Common Pitfalls & Decisions

**Correlation ≠ Causation:**
- High correlation doesn't mean one causes the other
- Both might be driven by a third factor
- Example: `hour` and `temperature` correlate (sun warms during day), but neither causes the other

**Non-Linear Relationships:**
- Correlation only measures **linear** relationships
- A curved relationship might show weak correlation
- Always visualize (scatter plots) to confirm

**Sample Size Matters:**
- Small datasets → Correlations less reliable
- Large datasets → Even tiny correlations can be statistically significant but not practically meaningful

**Categorical Variables:**
- This heatmap only includes numeric features
- `station_id` (if numeric) might show spurious correlation
- Use different methods for categorical analysis

---

### 🧠 Strategic Thinking

**Based on this heatmap, you should decide:**

1. **Feature Selection:**
   - Keep: Strong target correlation (>0.3) OR domain important
   - Remove: Redundant pairs (>0.9) OR weak + irrelevant (<0.1)
   - Investigate: Unexpected correlations

2. **Model Choice:**
   - Many correlated predictors → Ridge/Lasso or tree-based
   - Independent predictors → Any model works
   - Weak correlations → Need non-linear models

3. **Feature Engineering:**
   - Strong pairs → Don't create interactions (redundant)
   - Weak pairs with domain logic → Create interactions
   - Non-linear patterns → Add polynomial features

4. **Next Steps:**
   - Investigate surprising correlations (why?)
   - Check scatter plots for non-linear relationships
   - Consider time-lagged correlations (bikes 1 hour ago)

**Remember:** This heatmap is a **starting point** for understanding your data structure. It reveals linear relationships but can't show everything—interactions, non-linear patterns, and temporal dependencies require deeper analysis! 🔍

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. CORRELATION HEATMAP
# ═══════════════════════════════════════════════════════════

# Select numerical columns
numerical_cols = ['bikes_available', 'temperature', 'precipitation', 
                  'windspeed', 'hour', 'day_of_week', 'is_weekend']

# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Key correlations with bikes_available:")
for col in numerical_cols:
    if col != 'bikes_available':
        corr_value = correlation_matrix.loc['bikes_available', col]
        print(f"   {col:20s}: {corr_value:6.3f}")

### 🤔 Reflection: Correlation Heatmap

**Stop and think! Answer these questions before moving on:**

**1. Why did I create this visualization?**
<details>
<summary>Click to see example answer</summary>

To see all pairwise correlations at once and identify which features relate to our target (`bikes_available`) and to each other. This helps prioritize features and spot multicollinearity issues.
</details>

**2. What question does it answer?**
<details>
<summary>Click to see example answer</summary>

- Which features correlate most strongly with `bikes_available` (our target)?
- Are there redundant features (highly correlated with each other)?
- Are there unexpected correlations that suggest interesting relationships?
- Which features are independent (near-zero correlation)?
</details>

**3. What patterns am I looking for?**
<details>
<summary>Click to see example answer</summary>

- **Strong target correlations**: Dark red/blue in `bikes_available` row/column
- **Multicollinearity**: High correlations between predictors (redundant information)
- **Independence**: Near-zero correlations (features provide unique information)
- **Expected vs surprising**: Do correlations match domain intuition?
- **Sign**: Positive (both increase) vs negative (inverse relationship)
</details>

**4. How does this inform my modeling decisions?**
<details>
<summary>Click to see example answer</summary>

- **Feature selection**: Prioritize features with strong target correlation
- **Multicollinearity**: 
  - Problem for linear models (coefficients unstable)
  - Not a problem for tree-based models
  - Consider removing one of correlated pair or using Ridge regression
- **Feature engineering**: 
  - Weak correlations might be stronger with interactions
  - Non-linear transformations might reveal hidden patterns
- **Model choice**: 
  - If many intercorrelated features → Ridge/Lasso or tree-based
  - If clean independent features → simpler models work
- **Expectations**: What correlations are strongest? Match your hypothesis?
</details>

---

**✍️ Your observations** (write your insights here):
```
[Which features correlate most with bikes_available? Any surprising correlations? Multicollinearity issues?]






```

---

---

## 🎯 Part 6: Define the Prediction Problem

Now that we've explored the data, let's clearly define what we're trying to solve!

### 🔍 How Exploration Informed Our Problem Definition

**Before we dive into the problem statement, let's connect the dots!**

Every visualization and statistic we explored wasn't just busy work—each one directly shaped how we define this problem. Here's how:

---

#### 1. **Problem Statement** ← Time Series Visualization

**What we discovered:**
- Visualization 1 showed bike availability changing over time with clear patterns
- Observed stations reaching zero bikes (the business problem!)
- Patterns weren't random—they were predictable

**Impact on problem definition:**
- ✅ Confirmed the **business problem is real** (empty stations occur frequently)
- ✅ Patterns exist → This is **predictable**, not random
- ✅ Different stations behave differently → Need station-specific predictions

---

#### 2. **Problem Type: Regression** ← Descriptive Statistics

**What we discovered:**
- `df.describe()` showed `bikes_available` is continuous (0-20 range)
- Not discrete categories like "empty/full"
- Integer values with meaningful magnitudes

**Impact on problem definition:**
- ✅ Target is **continuous/numerical** → This is regression, not classification
- ✅ Must predict **specific numbers** (not just categories)
- ✅ Errors have magnitude (off by 2 bikes vs 10 bikes matters differently)

---

#### 3. **Temporal Features** ← Hourly Patterns & Weekday/Weekend

**What we discovered:**
- Visualization 2: Massive differences by hour (peak vs off-peak)
- Visualization 4: Clear weekday vs weekend patterns
- Hourly patterns showed non-linear relationships

**Impact on problem definition:**
- ✅ **Hour is critical** → Must include temporal features
- ✅ `is_weekend` matters → Added to feature list
- ✅ Time-of-day drives availability → These will be top predictors
- ✅ Identified need for future **temporal feature engineering** (lags, rolling averages in Module 04)

---

#### 4. **Weather Features** ← Temperature Scatter & Correlation Heatmap

**What we discovered:**
- Visualization 3: Temperature shows moderate relationship with availability
- Correlation heatmap: Weather correlations weaker than temporal
- Patterns exist but secondary to time

**Impact on problem definition:**
- ✅ Weather features are **relevant but secondary**
- ✅ Include `temperature`, `precipitation`, `windspeed`
- ✅ Set realistic expectations: Weather won't be the main driver
- ✅ Consider weather × time interactions in future modules

---

#### 5. **Station Features** ← Time Series & Categorical Analysis

**What we discovered:**
- Each station showed different patterns in time series plot
- `.nunique()` revealed multiple stations with distinct behaviors
- Location affects usage patterns

**Impact on problem definition:**
- ✅ **Station identity matters** → Include `station_id`
- ✅ Location coordinates might help → Include `latitude`/`longitude`
- ✅ May need station-specific models or station × time interactions

---

#### 6. **Target Variable Details** ← All Visualizations + Statistics

**What we discovered:**
- Range: 0 to ~20 bikes (from `df.describe()`)
- Integer values (from data inspection)
- Zero bikes observed frequently (from time series)
- Distribution shape understood (from box plots)

**Impact on problem definition:**
- ✅ Range: **0 to station capacity** → Bounded prediction problem
- ✅ **Can reach 0** → This is the critical business scenario!
- ✅ Integer values → May round predictions, treat edge cases carefully
- ✅ Goal: **Prevent empty stations** → Prioritize accuracy at low availability

---

#### 7. **Success Metrics** ← Descriptive Statistics + Domain Context

**What we discovered:**
- Mean ~11 bikes, std shows variability
- Scale of values makes certain errors more/less acceptable
- Business cares about predicting shortages

**Impact on problem definition:**
- ✅ **MAE** chosen → Interpretable ("off by X bikes")
- ✅ **RMSE** chosen → Penalizes big errors (critical for business)
- ✅ **R²** chosen → Measures overall predictive power
- ✅ Business metrics focus on empty station prevention

---

#### 8. **Baseline Performance** ← Descriptive Statistics

**What we discovered:**
- Mean bikes available: ~11
- Simple "predict the mean" strategy
- Variability in the data

**Impact on problem definition:**
- ✅ **Baseline = mean prediction** → Sets concrete bar to beat
- ✅ Baseline MAE/RMSE calculated → Quantified improvement target
- ✅ Any ML model must outperform this naive approach
- ✅ Provides context for evaluating "good" performance

---

#### 9. **Feature Prioritization** ← Correlation Heatmap

**What we discovered:**
- Which features correlate most with target
- Multicollinearity between predictors
- Redundant vs independent features

**Impact on problem definition:**
- ✅ Feature priority: **Hour > Station > Weather**
- ✅ Identified features to avoid (`docks_available` = data leakage)
- ✅ Informed feature engineering strategy for Module 04
- ✅ Guided model selection (tree-based models handle multicollinearity)

---

### 💡 Why This Matters

**Without exploration, we couldn't define:**
- ❌ Whether this is classification or regression
- ❌ Which features actually matter
- ❌ What "good performance" looks like
- ❌ Whether the problem is even predictable
- ❌ The scale and nature of predictions
- ❌ Business priorities (empty stations are the key issue)

**The exploration was essential because it:**
1. **Validated the problem exists** (empty stations happen)
2. **Proved it's solvable** (patterns exist, not random)
3. **Identified key features** (temporal > station > weather)
4. **Set realistic expectations** (baseline to beat)
5. **Informed modeling strategy** (regression, temporal features, tree-based models)

**This is why data scientists ALWAYS explore before modeling!** You can't define what you're solving until you understand what you have. 🔍 → 🎯

---

### The Problem Statement

**Business Goal:**
> Predict the number of available bikes at each station to improve user experience and optimize rebalancing operations.

**Machine Learning Task:**
> Given historical bike availability, weather conditions, and temporal features, predict `bikes_available` for a specific station and time.

---

### Problem Type

This is a **Regression** problem because:
- ✅ Target variable (`bikes_available`) is continuous/numerical
- ✅ We want to predict a specific number (not a category)
- ✅ Predictions can be evaluated with metrics like MAE, RMSE, R²

---

### Features (Input Variables)

**Temporal Features:**
- `timestamp` - Date and time
- `hour` - Hour of day (0-23)
- `day_of_week` - Day of week (0-6)
- `is_weekend` - Weekend indicator

**Weather Features:**
- `temperature` - Temperature in °C
- `precipitation` - Rainfall in mm
- `windspeed` - Wind speed in km/h

**Station Features:**
- `station_id` - Station identifier
- `station_name` - Station name
- `latitude` / `longitude` - Location

**Future Features (Module 04):**
- Lag features (previous bike counts)
- Rolling statistics (moving averages)
- Holiday indicators
- Points of interest nearby
- And more!

---

### Target Variable (Output)

**`bikes_available`** - Number of bikes available at a station

- Range: 0 to station capacity (e.g., 20)
- Integer values
- Can be 0 (problem for users!)
- Goal: Predict accurately to prevent empty stations

---

### Success Metrics

**Primary Metrics:**
1. **MAE (Mean Absolute Error)** - Average prediction error in bikes
   - Easy to interpret: "We're off by X bikes on average"
   - Goal: Minimize MAE

2. **RMSE (Root Mean Squared Error)** - Penalizes large errors more
   - Useful for catching big mistakes
   - Goal: Minimize RMSE

3. **R² (R-squared)** - Proportion of variance explained
   - Range: 0 to 1 (higher is better)
   - Goal: Maximize R²

**Business Metrics:**
- Reduction in "empty station" incidents
- User satisfaction scores
- Operational cost savings from better rebalancing

---

### Baseline Performance

What would a "naive" prediction achieve?


In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. BASELINE METRICS
# ═══════════════════════════════════════════════════════════

print("🎯 Baseline Performance (Predicting the Mean):\n")

# Simple baseline: always predict the mean
mean_bikes = df['bikes_available'].mean()
baseline_predictions = [mean_bikes] * len(df)

# Calculate baseline MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_mae = mean_absolute_error(df['bikes_available'], baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(df['bikes_available'], baseline_predictions))
baseline_r2 = r2_score(df['bikes_available'], baseline_predictions)

print(f"   Always predicting: {mean_bikes:.1f} bikes")
print(f"   Baseline MAE:  {baseline_mae:.2f} bikes")
print(f"   Baseline RMSE: {baseline_rmse:.2f} bikes")
print(f"   Baseline R²:   {baseline_r2:.3f}")

print("\n" + "="*60)
print("📊 THESE ARE THE NUMBERS TO BEAT!")
print("="*60)
print("\n⚠️  NOTE: We have NOT trained a model yet!")
print("   This is Module 01 (Exploration). Model training happens in Module 05.\n")
print("💡 When we train our ML model (Module 05), it MUST beat these metrics:")
print(f"   ✅ Target MAE:  < {baseline_mae:.2f} bikes  (lower is better)")
print(f"   ✅ Target RMSE: < {baseline_rmse:.2f} bikes  (lower is better)")
print(f"   ✅ Target R²:   > 0.000  (higher is better, max = 1.0)")
print("\n📌 If our model can't beat 'always predict the mean', it's useless!")

---

## 🚀 Part 7: Project Goals & Next Steps

### Project Goals

By the end of this project, we will:

1. **✅ Data Collection** (Module 02)
   - Fetch real-time data from bike and weather APIs
   - Store data efficiently
   - Handle API errors and rate limits

2. **✅ Data Exploration** (Module 03)
   - Automated profiling
   - Deep dive into patterns
   - Data quality assessment

3. **✅ Feature Engineering** (Module 04)
   - Create lag features
   - Rolling statistics
   - Holiday effects
   - Geographic features

4. **✅ Model Training** (Module 05)
   - Try multiple algorithms
   - Hyperparameter tuning
   - Model comparison

5. **✅ Validation** (Module 06)
   - Proper train/test splits
   - Cross-validation
   - Performance evaluation

6. **✅ Visualization** (Module 07)
   - Interactive dashboard
   - Business reporting
   - Insight communication

7. **✅ Automation** (Module 08)
   - Reproducible pipelines
   - Scheduled runs

8. **✅ Experimentation** (Module 09)
   - Track experiments with MLflow
   - A/B testing concepts

9. **✅ Collaboration** (Module 10)
   - Git workflows
   - Code review
   - Deployment preparation

10. **✅ Capstone**
    - Complete end-to-end system
    - Portfolio-ready project

---

### Success Criteria

**Module 01 Complete When:**
- [ ] ✅ You understand the bike-sharing problem
- [ ] ✅ You can load and explore data
- [ ] ✅ You can create basic visualizations
- [ ] ✅ You can clearly state the prediction problem
- [ ] ✅ You know what features and target we'll use
- [ ] ✅ You understand our success metrics
- [ ] ✅ You have baseline metrics to beat

**Are you ready to move to Module 02?** 🚀

---

## 🤔 Reflection Questions

Before finishing this module, reflect on these questions:

1. **What patterns did you observe in the data?**
   - Temporal patterns (hourly, weekday vs weekend)
   - Weather relationships
   - Station differences

2. **What surprised you?**
   - Empty stations happening frequently?
   - Correlation strengths?
   - Differences between stations?

3. **What features might be most important for prediction?**
   - Hour of day?
   - Temperature?
   - Station location?

4. **What additional data might help?**
   - Holidays?
   - Events?
   - Public transit schedules?

**Action**: Write your thoughts in a markdown cell below!

---

## ✅ Summary

**What you accomplished in this notebook:**

✔ **Loaded real data** - Bike availability + weather combined  
✔ **Inspected structure** - Understanding columns and data types  
✔ **Calculated statistics** - Mean, min, max, correlations  
✔ **Created visualizations** - Time series, hourly patterns, weather relationships  
✔ **Identified patterns** - Temporal and weather-driven behaviors  
✔ **Defined the problem** - Regression task with clear target and features  
✔ **Set goals** - Success metrics and baseline to beat  
✔ **Prepared for next steps** - Ready for Module 02!  

---

## 📊 Key Insights from Sample Data

1. **Temporal Patterns Are Strong**
   - Clear differences by hour of day
   - Weekday vs weekend variations
   - Rush hour effects visible

2. **Weather Matters**
   - Temperature correlates with availability
   - Precipitation impacts usage
   - Wind speed relevant

3. **Station Differences**
   - Different usage patterns
   - Location matters (train station vs tourist area)

4. **Empty Stations Happen**
   - Business problem is real!
   - Prediction can prevent this

---

## 🚀 Next Steps

**Congratulations!** You've completed Module 01! 🎉

**Next Module**: Module 02 - Data Acquisition
- Fetch real-time data from APIs
- Handle authentication and rate limits
- Store data efficiently
- Build data collection pipelines

**What to do now:**
1. ✅ Review the Module 01 README checklist
2. ✅ Ensure all notebooks run without errors
3. ✅ Read through any documentation links you skipped
4. ✅ Feel proud - you've built a strong foundation!

---

## 📚 Resources

- 📓 [Module 01 README](README.md) - Full module guide
- 🌐 [Open Data Sources](../../docs/reference/open_data_sources.md) - Data catalog
- 📊 [Example EDA Notebook](../example_data_exploration.ipynb) - More techniques
- 📐 [Coding Standards](../../docs/standards/coding_standards.md) - Best practices
- 📚 [Code Snippets](../../docs/standards/code_snippets.md) - Quick reference

---

**You're ready for Module 02!** Let's go fetch some real data! 🚀